# Evaluación de la Enfermedad Tiroidea: Elegir la Métrica Correcta
En el Machine Learning médico, seleccionar la métrica de evaluación correcta es tan crítico como elegir la arquitectura del modelo correcta. Este conjunto de datos presenta un desafío clásico y severo: **desequilibrio extremo de clases**.

Comencemos desde cero y entendamos cómo formular matemáticamente nuestras prioridades para detectar la enfermedad tiroidea.

## Por Qué la Precisión Es Engañosa
En nuestro conjunto de datos, aproximadamente el **90% de los pacientes son "negativos"** (sanos), mientras que solo el 10% pertenece a las clases "hipertiroideo" o "hipotiroideo". Si usamos la precisión estándar para evaluar nuestros modelos, nos topamos con el **problema del "Clasificador Ingenuo"**.

In [ ]:
import numpy as npfrom sklearn.metrics import accuracy_score# Imagine a test conjunto of 1000 patients# 900 negative (0), 50 hyperthyroid (1), 50 hypothyroid (2)y_true = np.array([0]*900 + [1]*50 + [2]*50)# A naive modelo that predicts EVERYONE is healthy ("negative")y_pred_naive = np.zeros(1000)accuracy = accuracy_score(y_true, y_pred_naive)print(f"Naive Model Accuracy: {accuracy * 100:.2f}%")

Naive Model Accuracy: 90.00%


Este modelo logra el 90% de precisión pero es **clínicamente inútil** porque no detectó a ningún paciente enfermo. La precisión recompensa al modelo simplemente por reconocer la distribución de la mayoría.

## Prioridades Clínicas: El Costo de los Errores
Para ir más allá de la precisión, debemos sopesar el costo real de los diferentes errores:

1. **Falso Negativo (Enfermedad No Detectada):** Un paciente tiene una condición tiroidea pero el modelo predice "negativo". Queda sin tratamiento, lo que puede llevar a complicaciones de salud graves, potencialmente fatales. **Costo: Extremadamente Alto**.
2. **Falso Positivo (Falsa Alarma):** Un paciente sano es marcado como enfermo. En este dominio, esto simplemente significa que el médico que recibe el resultado ordena un análisis de sangre secundario que vuelve normal. **Costo: Bajo** (comparativamente hablando).

Dado que nuestro objetivo principal es minimizar los Falsos Negativos, nuestra prioridad se convierte en **Recall (Sensibilidad)**. El Recall mide: *"De todos los pacientes que realmente tienen la enfermedad, ¿a cuántos identificamos con éxito?"*

## El Enfoque de Recall Puro
Reconociendo la necesidad de alta sensibilidad, podríamos considerar una métrica personalizada como el **Recall Medio de Tiroides**, definido como el promedio del recall de las dos clases de enfermedad:

$$\text{Score} = \frac{\text{Recall}_{\text{hipertiroideo}} + \text{Recall}_{\text{hipotiroideo}}}{2}$$

Esta métrica prioriza fuertemente encontrar pacientes enfermos. **Sin embargo**, introduce una trampa matemática conocida como el **"Problema del Modelo Trivial"**. Si un evaluador solo considera el Recall, un proceso de optimización (como `GridSearchCV`) inevitablemente lo explotará ignorando la Precisión por completo.

In [ ]:
from sklearn.metrics import recall_score# The "Trivial Panic modelo" - predicts everyone is sick!# Let's say it flips a coin to predict either hyperthyroid (1) or hypothyroid (2) for EVERY patientnp.random.seed(42)y_pred_panic = np.random.choice([1, 2], size=1000)# Calculate media Thyroid Recall for this conceptually broken modelohyper_recall = recall_score(y_true, y_pred_panic, labels=[1], average=None)[0]hypo_recall = recall_score(y_true, y_pred_panic, labels=[2], average=None)[0]mean_recall = (hyper_recall + hypo_recall) / 2print(f"Hyperthyroid Recall: {hyper_recall * 100:.2f}%")print(f"Hypothyroid Recall: {hypo_recall * 100:.2f}%")print(f"Thyroid Mean Recall Score: {mean_recall * 100:.2f}%")

Hyperthyroid Recall: 54.00%
Hypothyroid Recall: 60.00%
Thyroid Mean Recall Score: 57.00%


Observa que nuestro modelo "Pánico" logra un ~50% de Recall Medio artificialmente inflado. Matemáticamente, un modelo ignorante que predice ciegamente "enfermo" para todos siempre promediará un 50% de Recall Medio (una probabilidad $p$ de adivinar hipertiroideo y $1-p$ de adivinar hipotiroideo promedia $\frac{p + (1 - p)}{2} = 0.5$). Alcanza este "libre" 50% de base sin aprender absolutamente nada, simplemente clasificando mal a los **900 pacientes sanos**. Si bien los Falsos Positivos son "más baratos", no son gratuitos. Una métrica basada *puramente* en recall alienta a un optimizador a ignorar completamente la clase mayoritaria sana para obtener esa base garantizada. Pero un modelo con precisión cercana al 0% no tiene valor como herramienta de triaje, porque habría que hacer costosas pruebas de seguimiento a toda la población sana de todas formas. **No podemos ignorar la Precisión por completo.**

## La Solución: Equilibrando Trade-offs con $F_\beta$
Necesitamos una métrica que priorice agresivamente el Recall sin dejar que la Precisión caiga a cero. La puntuación $F_1$ estándar es la media armónica de Precisión y Recall, pero las pondera igual.

Para el cribado médico, usamos la **Puntuación $F_\beta$**, que pondera explícitamente el Recall como $\beta$ veces más importante que la Precisión. Al usar una **Macro $F_2$ Score** sobre solo nuestras dos clases de enfermedad, imponemos matemáticamente que encontrar a un paciente enfermo es el doble de importante que evitar un falso positivo.

In [ ]:
from sklearn.metrics import fbeta_score# Comparing our modelos with the robust Macro F2 score targeting ONLY the disease classes (1 and 2)def print_f2(model_name, y_predictions):    f2 = fbeta_score(y_true, y_predictions, beta=2, labels=[1, 2], average='macro', zero_division=0)    print(f"{model_name} Disease F2 Score: {f2:.4f}")print_f2("Naive Model", y_pred_naive)print_f2("Panic Model", y_pred_panic)# A good modelo that catches 90% of sicknesses with 50% precisiony_pred_good = y_true.copy()# Add some false positives (50 healthy people predicted as sick)y_pred_good[0:25] = 1y_pred_good[25:50] = 2# It misses a few (5 hyper and 5 hypo predicted as healthy)y_pred_good[-55:-50] = 0y_pred_good[-5:-1] = 0print_f2("Clinically Useful Model", y_pred_good)

Naive Model Disease F2 Score: 0.0000
Panic Model Disease F2 Score: 0.2035
Clinically Useful Model Disease F2 Score: 0.8410


## Implementación del Evaluador
Para usar correctamente la puntuación F₂ en `cross_val_score` y `RandomizedSearchCV`, la envolvemos con `make_scorer`. El `thyroid_scorer` resultante se exporta desde `src/metrics.py` y se usa **en todos los notebooks posteriores** como la única métrica de evaluación para este proyecto.

In [ ]:
from sklearn.metrics import make_scorerdef thyroid_disease_f2_score(y_true, y_pred):    """    Macro F2-score targeting strictly the two thyroid disease classes.    Handles both string labels and integer-encoded labels (alphabetical LabelEncoder ordering:    hyperthyroid=0, hypothyroid=1, negative=2).    """    y_true = np.asarray(y_true)    if np.issubdtype(y_true.dtype, np.integer) or (len(y_true) > 0 and isinstance(y_true[0], (int, np.integer))):        h_label, l_label = 0, 1    else:        h_label, l_label = 'hyperthyroid', 'hypothyroid'    return float(fbeta_score(y_true, y_pred, beta=2, labels=[h_label, l_label], average='macro', zero_division=0))# This scorer is what we importar as `thyroid_scorer` from src/metrics.pythyroid_scorer = make_scorer(thyroid_disease_f2_score)print("Scorer successfully created:", type(thyroid_scorer))

Scorer successfully created: <class 'sklearn.metrics._scorer._Scorer'>
